# Notebook 03 — Modeling

**Phase CRISP-DM : 4 sur 6**

Objectif : construire le moteur de recommandation content-based et le valider qualitativement.

### Approche technique
1. **Vectorisation** TF-IDF du champ `texte_modele` (uni+bigrams, min_df=2)
2. **Similarité** cosinus entre les vecteurs
3. **Ranking** top-k par score décroissant
4. **Test qualitatif** sur 3 requêtes représentatives

### Justification des choix
- **TF-IDF vs Bag-of-Words** : TF-IDF pondère les termes rares (noms de quartiers) et réduit le poids des mots fréquents (chambre, salon) → meilleure discrimination.
- **Uni + bigrams** : capture les expressions comme "chambre salon", "à louer", "a vendre".
- **Cosine similarity** : robuste à la longueur variable des documents.

In [1]:
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("../data/processed/annonces_clean.csv")
print(f"Shape : {df.shape}")
df[["id", "titre", "type_bien", "type_transaction", "ville_norm"]].head(3)

Shape : (255, 12)


,id,titre,type_bien,type_transaction,ville_norm
0,fb_69a97f6185f1dd70f9aef290,CHAMBRE SALON NOUVELLE CONSTRUCTION À LOUER À ...,chambre_salon,location,Lomé
1,fb_69a97f6185f1dd70f9aef291,A louer Chambre salon wc douche ( SANGUERA KLI...,chambre_salon,location,Lomé
2,fb_69a97f6185f1dd70f9aef292,A louer deux chambre salon agoè daliko,chambre_salon,location,Lomé


## 1. Vectorisation TF-IDF

In [2]:
FRENCH_STOPWORDS = [
    "le", "la", "les", "de", "des", "du", "un", "une", "et", "ou", "a", "au",
    "aux", "en", "pour", "par", "avec", "sans", "dans", "sur", "sous",
    "c est", "cest", "ce", "ces", "cette", "je", "tu", "il", "elle", "nous",
    "vous", "ils", "elles", "pas", "ne", "mais", "si", "que", "qui", "quoi",
    "son", "sa", "ses", "notre", "votre", "leur", "leurs", "mon", "ma", "mes",
]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    stop_words=FRENCH_STOPWORDS,
    sublinear_tf=True,
)
X = vectorizer.fit_transform(df["texte_modele"].fillna(""))
print(f"Matrice TF-IDF : {X.shape}  (sparsité : {100 * (1 - X.nnz / (X.shape[0] * X.shape[1])):.2f}%)")
print(f"Taille du vocabulaire : {len(vectorizer.vocabulary_)}")

Matrice TF-IDF : (255, 312)  (sparsité : 96.14%)
Taille du vocabulaire : 312


/sessions/happy-sleepy-lamport/.local/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['est'] not in stop_words.
  warnings.warn(


## 2. Matrice de similarité cosinus

In [3]:
sim_matrix = cosine_similarity(X)
np.fill_diagonal(sim_matrix, 0)  # exclure l'annonce elle-même des recommandations
print(f"Matrice sim : {sim_matrix.shape}")
print(f"Similarité moyenne : {sim_matrix.mean():.4f}")
print(f"Similarité max     : {sim_matrix.max():.4f}")

Matrice sim : (255, 255)
Similarité moyenne : 0.1059
Similarité max     : 1.0000


## 3. Fonction de recommandation (modèle hybride)

On combine :
- un **filtre dur** sur le type_bien et le type_transaction (logique métier)
- un **ranking souple** par similarité cosinus sur les vecteurs TF-IDF

C'est un modèle **hybride** : rules-based pour garantir la pertinence catégorielle, content-based pour ranker la similarité fine (quartier, surface, etc.).

In [4]:
def recommend(item_id: str, top_k: int = 5,
              filtre_transaction: bool = True,
              filtre_type_bien: bool = True):
    '''Retourne les top_k annonces les plus similaires à item_id.

    - filtre_transaction : restreint aux annonces de même type_transaction
    - filtre_type_bien   : restreint aux annonces de même type_bien
    Les deux filtres combinés forment la composante 'rules-based' du modèle hybride.
    '''
    if item_id not in df["id"].values:
        raise ValueError(f"id inconnu : {item_id}")
    idx = df.index[df["id"] == item_id][0]
    scores = sim_matrix[idx].copy()

    mask = np.ones(len(df), dtype=bool)
    if filtre_transaction:
        mask &= (df["type_transaction"] == df.loc[idx, "type_transaction"]).values
    if filtre_type_bien:
        mask &= (df["type_bien"] == df.loc[idx, "type_bien"]).values
    mask[idx] = False  # ne pas se recommander soi-même

    # Score -1 pour tout ce qui est hors filtre, afin que np.argsort les reléguée en fin
    scores = np.where(mask, scores, -1.0)

    top_idx = np.argsort(scores)[::-1][:top_k]
    # Si jamais moins de top_k candidats passent le filtre, on coupe là où le score devient -1
    top_idx = [j for j in top_idx if scores[j] >= 0]

    if not top_idx:
        return pd.DataFrame(columns=["id", "titre", "ville_norm", "quartier",
                                     "type_bien", "type_transaction", "prix", "score"])

    result = df.loc[top_idx, ["id", "titre", "ville_norm", "quartier",
                              "type_bien", "type_transaction", "prix"]].copy()
    result["score"] = np.array(scores)[top_idx].round(4)
    return result.reset_index(drop=True)

## 4. Tests qualitatifs

In [5]:
# Test 1 : une location chambre-salon à Lomé
test_id = df[(df["type_bien"] == "chambre_salon") &
             (df["type_transaction"] == "location")]["id"].iloc[0]
print(f"Annonce de référence : {df.loc[df['id']==test_id, 'titre'].iloc[0]}")
print()
recommend(test_id, top_k=5)

Annonce de référence : CHAMBRE SALON NOUVELLE CONSTRUCTION À LOUER À AGOE ZOSSIMÉ 00228 93818957



,id,titre,ville_norm,quartier,type_bien,type_transaction,prix,score
0,fb_69a97f6285f1dd70f9aef2ac,Chambre salon WC douche externe personnel à lo...,Lomé,Agoè,chambre_salon,location,25000.0,0.4318
1,fb_69a97f6185f1dd70f9aef292,A louer deux chambre salon agoè daliko,Lomé,Agoè,chambre_salon,location,40000.0,0.3467
2,ask_69a97c3b85f1dd70f9aef279,"studio meublé,louer,lome,agoe",Lomé,Lomé,chambre_salon,location,0.0,0.3435
3,fb_69a97f6385f1dd70f9aef2b1,Chambre salon interne a louer a lomé agoe mina...,Lomé,Lomé,chambre_salon,location,35000.0,0.3347
4,ask_69a97c3b85f1dd70f9aef273,"chambre salon,louer,lome,adidogome atilamonou",Lomé,Lomé,chambre_salon,location,0.0,0.3145


In [6]:
# Test 2 : un terrain en vente
mask = (df["type_bien"] == "terrain") & (df["type_transaction"].isin(["vente", "terrain"]))
if mask.any():
    test_id = df.loc[mask, "id"].iloc[0]
    print(f"Annonce de référence : {df.loc[df['id']==test_id, 'titre'].iloc[0]}")
    print()
    display(recommend(test_id, top_k=5))

Annonce de référence : Terain 1/2 lot Djidjolé pharmacie Point E



,id,titre,ville_norm,quartier,type_bien,type_transaction,prix,score
0,coin_69a9781885f1dd70f9aef1d8,Terrain 1/2 lot Djidjolé,Lomé,Djidjolé,terrain,terrain,23000000.0,0.9669
1,coin_69a9781885f1dd70f9aef1f5,Terrain 393 m² - Lomé,Lomé,Lomé,terrain,terrain,400000000.0,0.3971
2,coin_69a9781885f1dd70f9aef202,Terrain 1/4 de lot angle rue à Adidogomé molo ...,Lomé,Adidogomé,terrain,terrain,8500000.0,0.3197
3,coin_69a9781885f1dd70f9aef1ef,Terrain 1 lot à Totsi Ecobanc,Lomé,Totsi,terrain,terrain,65000000.0,0.2883
4,coin_69a9781885f1dd70f9aef21e,Terrain 1 lot à Tokoin super taco,Lomé,Tokoin,terrain,terrain,80000000.0,0.2883


In [7]:
# Test 3 : une villa ou maison
mask = df["type_bien"].isin(["villa", "maison"])
if mask.any():
    test_id = df.loc[mask, "id"].iloc[0]
    print(f"Annonce de référence : {df.loc[df['id']==test_id, 'titre'].iloc[0]}")
    print()
    display(recommend(test_id, top_k=5))

Annonce de référence : villa meublée,louer,lome,baguida



,id,titre,ville_norm,quartier,type_bien,type_transaction,prix,score
0,ask_69a97c3b85f1dd70f9aef277,"villa meublée,louer,lome,avenou",Lomé,Lomé,villa,location,0.0,0.7420
1,ask_69a97c3b85f1dd70f9aef252,"villa meublée,louer,lome,agoe",Lomé,Lomé,villa,location,0.0,0.7407
2,ask_69a97c3b85f1dd70f9aef27d,"villa meublée,louer,lome,hedzranawoe",Lomé,Lomé,villa,location,0.0,0.7221
3,fb_69a97f8f85f1dd70f9aef2d4,Villa meublée à louer à AVEDJI,Lomé,Avédji,villa,location,700.0,0.6355
4,fb_69a97f8f85f1dd70f9aef2d6,Villa meublée à louer à Togo 2000,Inconnu,Inconnu,villa,location,450.0,0.6093


## 5. Sauvegarde du modèle

In [8]:
import os
os.makedirs("../models", exist_ok=True)

artifact = {
    "vectorizer": vectorizer,
    "sim_matrix": sim_matrix.astype("float32"),
    "ids": df["id"].tolist(),
    "meta": df[["id", "titre", "ville_norm", "quartier",
                "type_bien", "type_transaction", "prix",
                "lien", "image_url"]].to_dict(orient="records"),
}
joblib.dump(artifact, "../models/tfidf_model.joblib", compress=3)
print("Modèle sauvegardé : models/tfidf_model.joblib")
print(f"Taille : {os.path.getsize('../models/tfidf_model.joblib') / 1024:.1f} KB")

Modèle sauvegardé : models/tfidf_model.joblib
Taille : 190.4 KB


### Conclusion — livrables de la phase Modeling
- Modèle TF-IDF + similarité cosinus entraîné et sauvegardé.
- Fonction `recommend(item_id, top_k)` opérationnelle.
- Tests qualitatifs : les recommandations sont visuellement cohérentes (même type de bien + même transaction + souvent même quartier).

➡️ Passage à la phase 5 : **Evaluation** (notebook 04) pour mesurer formellement Precision@K et Recall@K.